# 02 · Voir les tables SQL et les features

Ce notebook montre, **pour un client précis**, les lignes brutes de chaque table d'historique
puis le **résumé** (les features) que calcule la requête SQL correspondante.

👉 Changez `CLIENT` dans la cellule suivante pour explorer un autre client.

In [1]:
import duckdb
import pandas as pd

from credit_risk_scoring.config import DUCKDB_PATH
from credit_risk_scoring.features import SQL_DIR, read_sql

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

con = duckdb.connect(str(DUCKDB_PATH), read_only=True)

def sql(query):
    """Exécute une requête SQL et affiche le résultat sous forme de tableau."""
    return con.execute(query).df()

CLIENT = 185175   # identifiant SK_ID_CURR du client à explorer

## 0. Les tables disponibles

`SHOW TABLES` liste les tables de la base DuckDB. Vous pouvez écrire **n'importe quelle requête SQL** dans `sql("...")`.

In [2]:
sql("SHOW TABLES")

,name
0,POS_CASH_balance
1,application_test
2,application_train
3,bureau
4,bureau_balance
5,credit_card_balance
6,installments_payments
7,previous_application


In [3]:
# Colonnes et types d'une table
sql("DESCRIBE installments_payments")

,column_name,column_type,null,key,default,extra
0,SK_ID_PREV,BIGINT,YES,None,None,None
1,SK_ID_CURR,BIGINT,YES,None,None,None
2,NUM_INSTALMENT_VERSION,DOUBLE,YES,None,None,None
3,NUM_INSTALMENT_NUMBER,BIGINT,YES,None,None,None
4,DAYS_INSTALMENT,DOUBLE,YES,None,None,None
5,DAYS_ENTRY_PAYMENT,DOUBLE,YES,None,None,None
6,AMT_INSTALMENT,DOUBLE,YES,None,None,None
7,AMT_PAYMENT,DOUBLE,YES,None,None,None


## 1. La demande de crédit du client

In [4]:
sql(f"""
    SELECT SK_ID_CURR, TARGET, DAYS_BIRTH, DAYS_EMPLOYED, AMT_INCOME_TOTAL, AMT_CREDIT, AMT_ANNUITY
    FROM application_train
    WHERE SK_ID_CURR = {CLIENT}
""")

,SK_ID_CURR,TARGET,DAYS_BIRTH,DAYS_EMPLOYED,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY
0,185175,0,-23196,365243,50400.0,454500.0,20151.0


Après nettoyage par `01_application.sql` (ratios ajoutés, anomalie corrigée, genre retiré) :

In [5]:
sql(f"""
    SELECT SK_ID_CURR, AGE_YEARS, DAYS_EMPLOYED, DAYS_EMPLOYED_ANOM,
           CREDIT_INCOME_RATIO, ANNUITY_INCOME_RATIO, CREDIT_TERM, CREDIT_GOODS_RATIO
    FROM ({read_sql('01_application.sql')})
    WHERE SK_ID_CURR = {CLIENT}
""")

,SK_ID_CURR,AGE_YEARS,DAYS_EMPLOYED,DAYS_EMPLOYED_ANOM,CREDIT_INCOME_RATIO,ANNUITY_INCOME_RATIO,CREDIT_TERM,CREDIT_GOODS_RATIO
0,185175,63.507187,<NA>,1,9.017857,0.399821,0.044337,1.0


## 2. Remboursements passés (`installments_payments`)

**Avant** : une ligne par paiement. `DAYS_INSTALMENT` = date prévue, `DAYS_ENTRY_PAYMENT` = date du paiement
(en jours avant la demande). Si `DAYS_ENTRY_PAYMENT` > `DAYS_INSTALMENT`, le paiement est en retard.

In [6]:
sql(f"""
    SELECT SK_ID_PREV, NUM_INSTALMENT_NUMBER, DAYS_INSTALMENT, DAYS_ENTRY_PAYMENT,
           DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT AS jours_de_retard,
           AMT_INSTALMENT, AMT_PAYMENT
    FROM installments_payments
    WHERE SK_ID_CURR = {CLIENT}
    ORDER BY SK_ID_PREV, NUM_INSTALMENT_NUMBER
""")

,SK_ID_PREV,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,jours_de_retard,AMT_INSTALMENT,AMT_PAYMENT
0,1751523,1,-1959.0,-1974.0,-15.0,10308.825,10308.825
1,1751523,2,-1929.0,-1942.0,-13.0,10308.825,10308.825
2,1751523,3,-1899.0,-1911.0,-12.0,10308.825,10308.825
3,1751523,4,-1869.0,-1886.0,-17.0,10308.825,10308.825
4,1751523,5,-1839.0,-1802.0,37.0,10308.825,735.300
5,1751523,5,-1839.0,-1851.0,-12.0,10308.825,9573.525
6,1751523,6,-1809.0,-1802.0,7.0,10308.825,10308.825
7,1751523,7,-1779.0,-1802.0,-23.0,10308.825,70.875
8,1751523,7,-1779.0,-1764.0,15.0,10308.825,10237.950
9,1751523,8,-1749.0,-1764.0,-15.0,10308.105,112.050


**Après** `04_installments.sql` : une seule ligne qui résume tout cet historique.

In [7]:
sql(f"SELECT * FROM ({read_sql('04_installments.sql')}) WHERE SK_ID_CURR = {CLIENT}")

,SK_ID_CURR,INST_N_INSTALMENTS,INST_LATE_RATIO,INST_LATE_30_RATIO,INST_MEAN_DAYS_LATE,INST_MAX_DAYS_LATE,INST_UNDERPAID_RATIO,INST_TOTAL_UNPAID,INST_LATE_RATIO_LAST_YEAR
0,185175,31,0.16129,0.032258,2.225806,37.0,0.0,0.0,0.043478


## 3. Crédits dans d'autres organismes (`bureau`)

In [8]:
sql(f"""
    SELECT SK_ID_BUREAU, CREDIT_ACTIVE, CREDIT_TYPE, DAYS_CREDIT, AMT_CREDIT_SUM, AMT_CREDIT_SUM_DEBT,
           AMT_CREDIT_SUM_OVERDUE
    FROM bureau
    WHERE SK_ID_CURR = {CLIENT}
""")

,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_TYPE,DAYS_CREDIT,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_OVERDUE
0,6023988,Closed,Consumer credit,-1193,360000.0,0.0,0.0
1,6023989,Closed,Consumer credit,-1491,387000.0,0.0,0.0
2,6023990,Active,Consumer credit,-238,126000.0,98770.5,0.0


Historique mensuel de ces crédits (`bureau_balance`) : nombre de mois par statut
(`0` = à jour, `1` à `5` = retard croissant, `C` = clôturé, `X` = inconnu).

In [9]:
sql(f"""
    SELECT bb.SK_ID_BUREAU, bb.STATUS, COUNT(*) AS nb_mois
    FROM bureau_balance bb
    JOIN bureau b USING (SK_ID_BUREAU)
    WHERE b.SK_ID_CURR = {CLIENT}
    GROUP BY ALL
    ORDER BY 1, 2
""")

,SK_ID_BUREAU,STATUS,nb_mois
0,6023988,0,10
1,6023988,X,1
2,6023989,0,10
3,6023989,X,1
4,6023990,0,8


In [10]:
sql(f"SELECT * FROM ({read_sql('02_bureau.sql')}) WHERE SK_ID_CURR = {CLIENT}")

,SK_ID_CURR,BUREAU_N_CREDITS,BUREAU_N_ACTIVE,BUREAU_ACTIVE_RATIO,BUREAU_N_BAD_DEBT,BUREAU_CREDIT_CARD_RATIO,BUREAU_TOTAL_CREDIT,BUREAU_TOTAL_DEBT,BUREAU_DEBT_CREDIT_RATIO,BUREAU_TOTAL_OVERDUE,BUREAU_MAX_DAYS_OVERDUE,BUREAU_MAX_AMT_OVERDUE,BUREAU_N_PROLONGATIONS,BUREAU_OLDEST_CREDIT_DAYS,BUREAU_LAST_CREDIT_DAYS,BUREAU_LATE_MONTHS_RATIO,BUREAU_WORST_STATUS
0,185175,3,1,0.333333,0,0.0,873000.0,98770.5,0.113139,0.0,0,NaN,0.0,-1491,-238,0.0,0


## 4. Demandes précédentes chez Home Credit (`previous_application`)

In [11]:
sql(f"""
    SELECT SK_ID_PREV, NAME_CONTRACT_TYPE, NAME_CONTRACT_STATUS, AMT_APPLICATION, AMT_CREDIT,
           CNT_PAYMENT, DAYS_DECISION
    FROM previous_application
    WHERE SK_ID_CURR = {CLIENT}
    ORDER BY DAYS_DECISION
""")

,SK_ID_PREV,NAME_CONTRACT_TYPE,NAME_CONTRACT_STATUS,AMT_APPLICATION,AMT_CREDIT,CNT_PAYMENT,DAYS_DECISION
0,1751523,Consumer loans,Approved,70821.0,62064.0,8.0,-1999
1,2490891,Cash loans,Approved,675000.0,790830.0,60.0,-343
2,1889778,Revolving loans,Approved,135000.0,135000.0,0.0,-343
3,2381667,Cash loans,Refused,900000.0,900000.0,36.0,-26


In [12]:
sql(f"SELECT * FROM ({read_sql('03_previous_application.sql')}) WHERE SK_ID_CURR = {CLIENT}")

,SK_ID_CURR,PREV_N_APPLICATIONS,PREV_APPROVED_RATIO,PREV_REFUSED_RATIO,PREV_CANCELED_RATIO,PREV_CREDIT_APPLICATION_RATIO,PREV_MEAN_CREDIT,PREV_MEAN_N_PAYMENTS,PREV_LAST_DECISION_DAYS,PREV_LAST_REFUSAL_DAYS
0,185175,4,0.75,0.25,0.0,1.011988,471973.5,26.0,-26,-26


## 5. Cartes de crédit (`credit_card_balance`)

Une ligne par mois. Utilisation du plafond = `AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL`.

In [13]:
sql(f"""
    SELECT SK_ID_PREV, MONTHS_BALANCE, AMT_BALANCE, AMT_CREDIT_LIMIT_ACTUAL,
           ROUND(AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0), 2) AS utilisation,
           AMT_DRAWINGS_ATM_CURRENT, SK_DPD
    FROM credit_card_balance
    WHERE SK_ID_CURR = {CLIENT}
    ORDER BY MONTHS_BALANCE DESC
    LIMIT 12
""")

,SK_ID_PREV,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,utilisation,AMT_DRAWINGS_ATM_CURRENT,SK_DPD
0,1889778,-1,102627.900,247500,0.41,0.0,0
1,1889778,-2,99384.795,247500,0.40,0.0,0
2,1889778,-3,109542.780,247500,0.44,0.0,0
3,1889778,-4,112008.780,225000,0.50,0.0,0
4,1889778,-5,114310.125,225000,0.51,0.0,0
5,1889778,-6,116640.495,225000,0.52,0.0,0
6,1889778,-7,115233.165,135000,0.85,112500.0,0
7,1889778,-8,95824.620,135000,0.71,90000.0,0
8,1889778,-9,0.000,135000,0.00,NaN,0
9,1889778,-10,0.000,135000,0.00,NaN,0


In [14]:
sql(f"SELECT * FROM ({read_sql('05_credit_card.sql')}) WHERE SK_ID_CURR = {CLIENT}")

,SK_ID_CURR,CC_N_CARDS,CC_MEAN_UTILIZATION,CC_MAX_UTILIZATION,CC_MEAN_ATM_DRAWINGS,CC_LATE_MONTHS_RATIO,CC_MAX_DPD
0,185175,1,0.395133,0.853579,25312.5,0.0,0


## 6. Crédits à la consommation (`POS_CASH_balance`)

In [15]:
sql(f"""
    SELECT SK_ID_PREV, MONTHS_BALANCE, CNT_INSTALMENT, CNT_INSTALMENT_FUTURE, NAME_CONTRACT_STATUS, SK_DPD
    FROM POS_CASH_balance
    WHERE SK_ID_CURR = {CLIENT}
    ORDER BY SK_ID_PREV, MONTHS_BALANCE DESC
    LIMIT 12
""")

,SK_ID_PREV,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD
0,1751523,-57,8.0,0.0,Completed,0
1,1751523,-58,8.0,1.0,Active,0
2,1751523,-59,8.0,2.0,Active,0
3,1751523,-60,8.0,3.0,Active,16
4,1751523,-61,8.0,4.0,Active,0
5,1751523,-62,8.0,5.0,Active,0
6,1751523,-63,8.0,6.0,Active,0
7,1751523,-64,8.0,7.0,Active,0
8,1751523,-65,8.0,8.0,Active,0
9,2490891,-7,4.0,0.0,Completed,0


In [16]:
sql(f"SELECT * FROM ({read_sql('06_pos_cash.sql')}) WHERE SK_ID_CURR = {CLIENT}")

,SK_ID_CURR,POS_N_LOANS,POS_LATE_MONTHS_RATIO,POS_MAX_DPD,POS_REMAINING_RATIO
0,185175,2,0.071429,16,0.0


## 7. Le résultat final : la ligne du client dans `features.parquet`

Toutes les features réunies sur **une seule ligne** : c'est ce que le modèle verra.

In [17]:
pd.set_option("display.max_rows", 200)   # afficher les 172 lignes
features = pd.read_parquet("../data/processed/features.parquet")
features.loc[features["SK_ID_CURR"] == CLIENT].T.rename(columns=lambda _: "valeur")

,valeur
SK_ID_CURR,185175
TARGET,0
NAME_CONTRACT_TYPE,Cash loans
FLAG_OWN_CAR,Y
FLAG_OWN_REALTY,N
CNT_CHILDREN,0
AMT_INCOME_TOTAL,50400.0
AMT_CREDIT,454500.0
AMT_ANNUITY,20151.0
AMT_GOODS_PRICE,454500.0


## 8. À vous de jouer

Quelques requêtes pour vous entraîner :

1. Combien de clients ont au moins un crédit en `Bad debt` dans `bureau` ?
2. Quel est le taux de défaut (`TARGET`) des clients qui ont déjà été refusés au moins une fois ?
3. Quels sont les 5 clients avec le plus d'échéances payées en retard ?

In [18]:
# Exemple pour la question 2
sql("""
    SELECT
        (p.SK_ID_CURR IS NOT NULL) AS deja_refuse,
        COUNT(*) AS nb_clients,
        ROUND(AVG(a.TARGET) * 100, 2) AS taux_defaut_pct
    FROM application_train a
    LEFT JOIN (SELECT DISTINCT SK_ID_CURR FROM previous_application
               WHERE NAME_CONTRACT_STATUS = 'Refused') p USING (SK_ID_CURR)
    GROUP BY 1
""")

,deja_refuse,nb_clients,taux_defaut_pct
0,False,207217,6.98
1,True,100294,10.32
